In [1]:
import os
from glob import glob
import json
from pathlib import Path
import mne
from mne.preprocessing import fix_stim_artifact, ICA
from mne_faster import find_bad_channels, find_bad_epochs

from mne_tesa.mne_tesa import tesa_interp_data_cubic_spline, tesa_replace_constant_amplitude, ICA_TESA, mne_tesa_class_comp, tesa_ica_select, notch_filter_epochs
from tesa_py import rename_channels

In [2]:
with open('file_list.json', 'r') as f:
    json_data = json.load(f)

# Create set of filenames to exclude (everything after 'xnat/')
exclude_files = {item['file'].split('xnat/', 1)[1] for item in json_data if 'xnat/' in item['file']}

path = '/run/media/alexe/T7_Shield/xnat'

# Build result list with exclusion filter built-in
result = [
    y for x in os.walk(path) 
    for y in glob(os.path.join(x[0], "*.dat"))
    if y.replace(path, '').lstrip('/') not in exclude_files
]

print(f"Loaded {len(result)} files (excluded {len(exclude_files)} from list)")

Loaded 646 files (excluded 292 from list)


In [3]:
with open(os.path.join(path, 'progress.json'), 'r') as f:
    last = json.load(f)
start_idx = last['index'] + 1 

print(f"Start index {start_idx}")

Start index 95


In [ ]:
#filename = result[0]
filename = result[start_idx]

#### Bad files:
GWM075_SP_DLPFC.dat

GWM064_SP_PP.dat

GWM009_SP_DLPFC.dat

#### Good files for demo
1. MBI_XNAT_E22764/GWM009_SP_PP/457411/GWM009_SP_PP.dat
2. MBI_XNAT_E22764/GWM009_SP_FEF/457427/GWM009_SP_FEF.dat

In [ ]:
filename

In [ ]:
raw = mne.io.read_raw_curry(filename, preload=True)
raw = rename_channels(raw)  # This function should be improved
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

event_descriptions = ["Response/R128", "Stimulus/A", "128", "191"]
events, event_dict = None, None

for desc in event_descriptions:
    try:
        events, event_dict = mne.events_from_annotations(raw, event_id={desc: 1})
        print(f"Successfully found events using description: '{desc}'")
        break
    except Exception:
        print(f"Could not find events for description: '{desc}'. Trying next...")

tmin_cut, tmax_cut = -0.002, 0.015

## Epoched with a window of -1000 ms to 1000 ms around the TMS pulse, and the mean of each channel’s baseline (defined as a window of -500 to -10ms)

In [ ]:
epochs = mne.Epochs(raw,
    events=events,
    tmin=-1,
    tmax=1,
    detrend=None,
    baseline=(-0.5, -0.01),
    preload=True,
)
# TMS pulse artefact (the data from -2 to 15ms) was then removed and a cubic interpolation was
# applied to replace the missing data.

epochs = tesa_interp_data_cubic_spline(epochs, inst_type="epochs", tmin=tmin_cut, tmax=tmax_cut, events=events)

## This seems shaky.. probably better not to do this
epochs.resample(1000)  # The recordings were then down sampled to 1000 Hz

print(
    f"\n \n REMOVED AND INTERPOLATED TMS PULSE BETWEEN {tmin_cut} and {tmax_cut} using cubic interpolation \n \n"
)

## Following which, the trials and channels affected by prominent artefacts were detected by visual inspections and removed
### MNE-FASTER instead

In [ ]:
epochs.info["bads"] = find_bad_channels(epochs, thres=3)
bad_channels = epochs.info["bads"]

if "Fp1" in bad_channels:
    bad_channels.remove('Fp1')
if "Fp2" in bad_channels:
    bad_channels.remove('Fp2')
if "F7" in bad_channels:
    bad_channels.remove('F7')
if "F8" in bad_channels:
    bad_channels.remove('F8')

bad_epochs = find_bad_epochs(epochs, thres=3)
epochs.drop(bad_epochs)

## Afterwards, the interpolated data around TMS pulse were replaced with constant amplitude data
epochs = tesa_replace_constant_amplitude(epochs, inst_type="epochs", tmin=tmin_cut, tmax=tmax_cut, events=events)

## The TMS induced muscle and decay artefacts were identified using the FastICA algorithm
### MNE automatically excludes bad channels from ICA

In [ ]:
ica1 = ICA_TESA(n_components=30, noise_cov=None, 
           random_state=42, method='fastica', 
           fit_params=None, max_iter='auto', 
           allow_ref_meg=False, verbose=False)

ica1.fit(epochs)

suggested, comp_df = mne_tesa_class_comp(
    epochs,
    ica1,
    tmsMuscle=True,
    tmsMuscleThresh=8,  # 8 was the default in TESA
    tmsMuscleWin=[0.011, 0.030],
    blink=True,
    blinkThresh=2.5,
    blinkElecs=["Fp1", "Fp2"],
    lat_eye_move=True,
    lat_eye_moveThresh=2.0,
    lat_eye_move_elecs=["F7", "F8"],
    persistant_muscle=True,
    persistant_muscle_thresh=0.5,  # original in TESA was -0.31,
    muscleFreqIn=[7, 70],  # original in TESA was [7, 70]
    # muscleFreqEx=[48, 52], ## no exclusion made in MNE implementation
    electrode_noise=True,
    elec_noise_thresh=4,
)

In [ ]:
exclude = tesa_ica_select(epochs, ica1, suggested)

In [ ]:
#exclude.pop(1)

In [ ]:
exclude

In [ ]:
ica1.exclude = exclude
ica1.apply(epochs)

In [ ]:
save_name = Path(filename).stem + 'ICA1_preproc-ica.fif'
ica1.save(f"/run/media/alexe/T7_Shield/xnat/xnat_preproc/{save_name}", overwrite=True)

## A linear interpolation was applied to replace the missing data.
Function already exists in MNE

In [ ]:
fix_stim_artifact(epochs, events=events, event_id=None, tmin=tmin_cut, tmax=tmax_cut, baseline=None, mode='linear', stim_channel=None, picks=None)

## Data were then band-pass (1-100 Hz) and band-stop (48-52 Hz) filtered using a zero-phase Butterworth filter (order = 4)

In [ ]:
epochs.filter(
        l_freq=1,
        h_freq=100,
        picks=None,
        filter_length="auto",
        l_trans_bandwidth="auto",
        h_trans_bandwidth="auto",
        n_jobs=-1,
        method="iir",
        iir_params=dict(order=4, ftype='butter', output='sos'),
        phase="zero",
        fir_window="hamming",
        fir_design="firwin",
        skip_by_annotation=("edge", "bad_acq_skip"),
        pad="reflect_limited",
        verbose=True,
    )
epochs = notch_filter_epochs(epochs)

### Other artefacts such as blinks and noise-related artefacts were corrected by applying a second run of FastICA algorithm. Artefactual components were selected using the TESA automatedfunctions with default settings and checked visually

In [ ]:
ica2 = ICA_TESA(n_components=30, noise_cov=None, 
           random_state=42, method='fastica', 
           fit_params=None, max_iter='auto', 
           allow_ref_meg=False, verbose=False)
ica2.fit(epochs)

suggested, comp_df2 = mne_tesa_class_comp(
    epochs,
    ica2,
    tmsMuscle=True,
    tmsMuscleThresh=8,  # 8 was the default in TESA
    tmsMuscleWin=[0.011, 0.030],
    blink=True,
    blinkThresh=2.5,
    blinkElecs=["Fp1", "Fp2"],
    lat_eye_move=True,
    lat_eye_moveThresh=2.0,
    lat_eye_move_elecs=["F7", "F8"],
    persistant_muscle=True,
    persistant_muscle_thresh=0.5,  # original in TESA was -0.31,
    muscleFreqIn=[7, 70],  # original in TESA was [7, 70]
    # muscleFreqEx=[48, 52], ## no exclusion made in MNE implementation
    electrode_noise=True,
    elec_noise_thresh=4,
)

In [ ]:
#suggested

In [ ]:
exclude2 = tesa_ica_select(epochs, ica2, suggested)

In [ ]:
#exclude2.pop(4)
#exclude2

In [ ]:
exclude2

In [ ]:
ica2.exclude = exclude2
ica2.apply(epochs)

In [ ]:
save_name = Path(filename).stem + 'ICA2_preproc-ica.fif'
ica2.save(f"/run/media/alexe/T7_Shield/xnat/xnat_preproc/{save_name}", overwrite=True)

## Finally, the rejected channels were spatially interpolated using spherical method and all data were re-referenced to the common average

In [ ]:
epochs.interpolate_bads()
epochs.set_eeg_reference('average')

In [ ]:
#evoked = epochs.average().plot();

In [ ]:
#epochs.average().plot(xlim=(-0.2, 0.3));

In [ ]:
from mne_tesa.plotly_evoked import plot_gfp, plot_evoked
plot_gfp(epochs.average(), epochs.average().times, xlim=(-0.05, 0.3))

In [ ]:
plot_evoked(epochs.average(), epochs.times)

In [ ]:
filename

In [ ]:
with open(os.path.join(path, 'progress.json'), 'w') as f:
    json.dump({'current': filename, 'index': result.index(filename)}, f)

In [ ]:
preproc_stats = {
    "filename": str(filename),
    "bad_channels": bad_channels,
    "n_bad_channels": len(bad_channels),
    "bad_epochs": bad_epochs.tolist() if hasattr(bad_epochs, 'tolist') else list(bad_epochs),
    "n_bad_epochs": len(bad_epochs),
    "ica1_excluded_components": exclude.tolist() if hasattr(exclude, 'tolist') else list(exclude),
    "n_ica1_excluded_components": len(exclude2),
    "ica2_excluded_components": exclude.tolist() if hasattr(exclude2, 'tolist') else list(exclude2),
    "n_ica2_excluded_components": len(exclude2)
}

In [ ]:
# Save epochs
save_name = Path(filename).stem + '_preproc-epo.fif'
epochs.save(f"/run/media/alexe/T7_Shield/xnat/xnat_preproc/{save_name}", overwrite=True)

In [ ]:
# Save preprocessing stats
stats_filename = Path(filename).stem + '_preproc_stats.json'
stats_path = f"/run/media/alexe/T7_Shield/xnat/xnat_preproc/{stats_filename}"

with open(stats_path, 'w') as f:
    json.dump(preproc_stats, f, indent=4)

print(f"Saved preprocessing stats to {stats_filename}")